In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [2]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [3]:
model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    temperature=0.3
)

In [4]:
from autogen_core.tools import FunctionTool
from langchain_tavily import TavilySearch

In [5]:
tavily_search = TavilySearch()

In [6]:
def search_web(query: str) -> dict:
    """
    Returns web search results for a given query.
    """
    try:
        results = tavily_search.invoke(query)
        return results
    except Exception:
        return {"error": "No results found."}

web_search_tool = FunctionTool(
    search_web,
    description="Performs a web search and returns relevant results for the given query."
)

In [7]:

def percentage_change_fn(start: float, end: float) -> float:
    """
    Returns the percentage change from start to end.
    """
    return ((end - start) / start) * 100

percentage_change_tool = FunctionTool(
    percentage_change_fn,
    description="Returns the percentage change from a starting value to an ending value."
)

In [8]:

planning_agent = AssistantAgent(
    name="PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

web_search_agent = AssistantAgent(
    name="WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""
    You are a web search agent.
    Your only tool is search_tool - use it to find information.
    You make only one search call at a time.
    Once you have the results, you never do calculations based on them.
    """,
)

data_analyst_agent = AssistantAgent(
    name="DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""
    You are a data analyst.
    Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
    If you have not seen the data, ask for it.
    """,
)

In [17]:
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.messages import TextMessage

In [10]:
text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=25)
termination = text_mention_termination | max_messages_termination

In [11]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [13]:
from typing import List, Sequence
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage

#### **Custom Selector Function**

In [14]:
def selector_func(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    if messages[-1].source != planning_agent.name:
        return planning_agent.name
    
    return None

In [15]:

team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,  
    selector_func=selector_func
)

In [16]:
task = """
Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
"""

# Use asyncio.run(...) if you are running this in a script.
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------

Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?

---------- TextMessage (PlanningAgent) ----------
To address your request, I will break down the tasks as follows:

1. WebSearchAgent : Find out which player from the Chennai Super Kings scored the most runs in the 2018 IPL season.
2. WebSearchAgent : Find the total dismissals of MS Dhoni in the 2019 IPL season.
3. WebSearchAgent : Find the total dismissals of MS Dhoni in the 2020 IPL season.
4. DataAnalystAgent : Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.

Once these tasks are completed, I will summarize the findings.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_dMBPkFbrbTdTJTTRqvUnsl4f', arguments='{"query": "Chennai Super Kings player most runs 2018 IPL season"}', 

TaskResult(messages=[TextMessage(id='3378e5b8-a03a-467a-907b-5da746a79791', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 24, 16, 34, 19, 638302, tzinfo=datetime.timezone.utc), content="\nWho was the Chennai Super Kings player with the most runs in the 2018 IPL season, \nand what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?\n", type='TextMessage'), TextMessage(id='4f3ed734-f726-4795-843d-eb2db358c851', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=160, completion_tokens=130), metadata={}, created_at=datetime.datetime(2025, 7, 24, 16, 34, 23, 884602, tzinfo=datetime.timezone.utc), content="To address your request, I will break down the tasks as follows:\n\n1. WebSearchAgent : Find out which player from the Chennai Super Kings scored the most runs in the 2018 IPL season.\n2. WebSearchAgent : Find the total dismissals of MS Dhoni in the 2019 IPL season.\n3. WebSearchAgent : Find the tot

In [18]:
async def run_team():
    task = TextMessage(content="""
                        Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
                        and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
                        """, source='user')

    result = await team.run(task=task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")


    # print(result)

await run_team()

user : 
                        Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
                        and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
                        
PlanningAgent : To address your request, I will break down the tasks as follows:

1. WebSearchAgent : Find out which player from the Chennai Super Kings scored the most runs in the 2018 IPL season.
2. WebSearchAgent : Find the total dismissals of MS Dhoni in the 2019 IPL season.
3. WebSearchAgent : Find the total dismissals of MS Dhoni in the 2020 IPL season.
4. DataAnalystAgent : Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.

Once these tasks are completed, I will summarize the findings.
WebSearchAgent : [FunctionCall(id='call_hvYvl7GtAgbZVmEqWujl53EC', arguments='{"query": "Chennai Super Kings player most runs 2018 IPL season"}', name='search_web'), FunctionCall(id='ca

#### **Custom Candidate Function**

In [19]:

def candidate_func(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> List[str]:
    # keep planning_agent first one to plan out the tasks
    if messages[-1].source == "user":
        return [planning_agent.name]

    # if previous agent is planning_agent and if it explicitely asks for web_search_agent
    # or data_analyst_agent or both (in-case of re-planning or re-assignment of tasks)
    # then return those specific agents
    last_message = messages[-1]
    if last_message.source == planning_agent.name:
        participants = []
        if web_search_agent.name in last_message.to_text():
            participants.append(web_search_agent.name)
        if data_analyst_agent.name in last_message.to_text():
            participants.append(data_analyst_agent.name)
        if participants:
            return participants  # SelectorGroupChat will select from the remaining two agents.

    # we can assume that the task is finished once the web_search_agent
    # and data_analyst_agent have took their turns, thus we send
    # in planning_agent to terminate the chat
    previous_set_of_agents = set(message.source for message in messages)
    if web_search_agent.name in previous_set_of_agents and data_analyst_agent.name in previous_set_of_agents:
        return [planning_agent.name]

    # if no-conditions are met then return all the agents
    return [planning_agent.name, web_search_agent.name, data_analyst_agent.name]

In [20]:
# Reset the previous team and run the chat again with the selector function.
await team.reset()

team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    candidate_func=candidate_func,
)

In [21]:
async def run_team():
    task = TextMessage(content="""
                        Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
                        and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
                        """, source='user')

    result = await team.run(task=task)

    for each_agent_message in result.messages:
        print(f"{each_agent_message.source} : {each_agent_message.content}")
        print('\n')


    # print(result)

await run_team()

user : 
                        Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
                        and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
                        


PlanningAgent : To address your request, I will break it down into two main subtasks:

1. Identify the Chennai Super Kings player with the most runs in the 2018 IPL season.
2. Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.

Now, I will assign these tasks to the appropriate agents:

1. WebSearchAgent : Find out who was the Chennai Super Kings player with the most runs in the 2018 IPL season.
2. DataAnalystAgent : Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.

Once these tasks are completed, I will summarize the findings.


WebSearchAgent : [FunctionCall(id='call_ZAALGpsWF0imkuxNC7XBjVsl', arguments='{"query": "Chenn

#### **User Feedback**

In [22]:
from autogen_agentchat.agents import UserProxyAgent

In [23]:
user_proxy_agent = UserProxyAgent(
    name="UserProxyAgent", 
    description="A proxy for the user to approve or disapprove tasks."
)

In [24]:
def selector_func_with_user_proxy(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    if messages[-1].source != planning_agent.name and messages[-1].source != user_proxy_agent.name:
        # Planning agent should be the first to engage when given a new task, or check progress.
        return planning_agent.name
    
    if messages[-1].source == planning_agent.name:
        if messages[-2].source == user_proxy_agent.name and "APPROVE" in messages[-1].content.upper():  # type: ignore
            # User has approved the plan, proceed to the next agent.
            return None
        # Use the user proxy agent to get the user's approval to proceed.
        return user_proxy_agent.name
    
    if messages[-1].source == user_proxy_agent.name:
        # If the user does not approve, return to the planning agent.
        if "APPROVE" not in messages[-1].content.upper():  # type: ignore
            return planning_agent.name
        
    return None

In [25]:
# Reset the previous agents and run the chat again with the user proxy agent and selector function.
await team.reset()

team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    selector_func=selector_func_with_user_proxy,
    allow_repeated_speaker=True,
)

In [26]:
task = """
Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
"""

# Use asyncio.run(...) if you are running this in a script.
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------

Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?

---------- TextMessage (PlanningAgent) ----------
To address your request, I will break it down into two main subtasks:

1. Find out who the Chennai Super Kings player with the most runs in the 2018 IPL season was.
2. Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.

Now, I will assign these tasks to the appropriate agents:

1. WebSearchAgent : Find out who the Chennai Super Kings player with the most runs in the 2018 IPL season was.
2. DataAnalystAgent : Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.
---------- TextMessage (UserProxyAgent) ----------
APPROVE
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_vCeyD8y6l4cWGDUv

TaskResult(messages=[TextMessage(id='34f34a8f-10cd-4ed4-ab98-9cebad6eb95d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 24, 16, 58, 53, 696575, tzinfo=datetime.timezone.utc), content="\nWho was the Chennai Super Kings player with the most runs in the 2018 IPL season, \nand what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?\n", type='TextMessage'), TextMessage(id='f4e31085-e89e-4873-8312-6ed70cae027f', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=160, completion_tokens=135), metadata={}, created_at=datetime.datetime(2025, 7, 24, 16, 58, 59, 636789, tzinfo=datetime.timezone.utc), content="To address your request, I will break it down into two main subtasks:\n\n1. Find out who the Chennai Super Kings player with the most runs in the 2018 IPL season was.\n2. Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.\n\nNow, I will assign these

#### **Using Reasoning Models**

In [27]:
model_client = OpenAIChatCompletionClient(model="o3-mini")

In [29]:
web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""Use web search tool to find information.""",
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""Use tool to perform calculation. If you have not seen the data, ask for it.""",
)

user_proxy_agent = UserProxyAgent(
    "UserProxyAgent",
    description="A user to approve or disapprove tasks.",
)

selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
When the task is complete, let the user approve or disapprove the task.
"""

team = SelectorGroupChat(
    participants=[web_search_agent, data_analyst_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=termination,  # Use the same termination condition as before.
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
)

In [30]:
task = """
Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
"""

await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------

Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?

---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_ZfzOqtrf4nUt2TTlG8yYKtvr', arguments='{"query": "Chennai Super Kings player with the most runs in the 2018 IPL season; MS Dhoni percentage change in total dismissals between 2019 and 2020 seasons?"}', name='search_web')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content='{\'query\': \'Chennai Super Kings player with the most runs in the 2018 IPL season; MS Dhoni percentage change in total dismissals between 2019 and 2020 seasons?\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'url\': \'https://en.wikipedia.org/wiki/Chennai_Super_Kings\', \'title\': \'Chennai Super Kings - Wikipedia\', \'content\': \'T

TaskResult(messages=[TextMessage(id='8dfc7d32-52c7-47ac-94d6-71fd523c6da8', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 24, 17, 13, 33, 459345, tzinfo=datetime.timezone.utc), content="\nWho was the Chennai Super Kings player with the most runs in the 2018 IPL season, \nand what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?\n", type='TextMessage'), ToolCallRequestEvent(id='b38771c3-f018-4c16-a670-c5ae0a7fc9a5', source='WebSearchAgent', models_usage=RequestUsage(prompt_tokens=114, completion_tokens=192), metadata={}, created_at=datetime.datetime(2025, 7, 24, 17, 13, 48, 201275, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_ZfzOqtrf4nUt2TTlG8yYKtvr', arguments='{"query": "Chennai Super Kings player with the most runs in the 2018 IPL season; MS Dhoni percentage change in total dismissals between 2019 and 2020 seasons?"}', name='search_web')], type='ToolCallRequestEvent'), ToolCallExecutio